## Problem statement is the best algorithm to predict user ratings for films
* Load dataset
* Require library
* data inspection
  


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/netflix-inc/netflix-prize-data/README
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_1.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_2.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_3.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_4.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/movie_titles.csv
/kaggle/input/datasets/netflix-inc/netflix-prize-data/probe.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/qualifying.txt


In [2]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/netflix-inc/netflix-prize-data/README
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_1.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_2.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_3.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_4.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/movie_titles.csv
/kaggle/input/datasets/netflix-inc/netflix-prize-data/probe.txt
/kaggle/input/datasets/netflix-inc/netflix-prize-data/qualifying.txt


In [3]:
path = '/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_1.txt'

with open(path, 'r') as f:
    for i in range(10):
        print(f.readline().strip())

1:
1488844,3,2005-09-06
822109,5,2005-05-13
885013,4,2005-10-19
30878,4,2005-12-26
823519,3,2004-05-03
893988,3,2005-11-17
124105,4,2004-08-05
1248029,3,2004-04-22
1842128,4,2004-05-09


In [4]:
import pandas as pd

path = '/kaggle/input/datasets/netflix-inc/netflix-prize-data/combined_data_1.txt'

rows = []
movie_id = None

with open(path, 'r') as f:
    for i, line in enumerate(f):
        if i >= 100000:
            break
        line = line.strip()
        if line.endswith(':'):
            movie_id = int(line[:-1])
        else:
            customer_id, rating, date = line.split(',')
            rows.append((movie_id, int(customer_id), int(rating), date))

df = pd.DataFrame(rows, columns=['movie_id', 'customer_id', 'rating', 'date'])
df['date'] = pd.to_datetime(df['date'])

print(df.shape)
df.head()

(99970, 4)


,movie_id,customer_id,rating,date
0,1,1488844,3,2005-09-06
1,1,822109,5,2005-05-13
2,1,885013,4,2005-10-19
3,1,30878,4,2005-12-26
4,1,823519,3,2004-05-03


In [5]:
df.isnull().sum()

movie_id       0
customer_id    0
rating         0
date           0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99970 entries, 0 to 99969
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   movie_id     99970 non-null  int64         
 1   customer_id  99970 non-null  int64         
 2   rating       99970 non-null  int64         
 3   date         99970 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(3)
memory usage: 3.1 MB


In [7]:
print(df['rating'].value_counts().sort_index())

rating
1     5919
2    10947
3    28608
4    33012
5    21484
Name: count, dtype: int64


In [8]:
movie_avg = df.groupby('movie_id')['rating'].agg(['count', 'mean'])
print(movie_avg)

          count      mean
movie_id                 
1           547  3.749543
2           145  3.558621
3          2012  3.641153
4           142  2.739437
5          1140  3.919298
6          1019  3.084396
7            93  2.129032
8         14910  3.189805
9            95  2.621053
10          249  3.180723
11          198  3.030303
12          546  3.417582
13          125  4.552000
14          118  3.025424
15          290  3.286207
16         2699  3.098555
17         7108  2.903208
18        10722  3.784369
19          539  3.324675
20          116  3.146552
21          218  3.463303
22          203  2.246305
23          615  3.556098
24         1333  2.993998
25         1207  3.970174
26         5861  2.793721
27          273  3.527473
28        39752  3.823254
29          523  3.598470
30         7172  3.758226


In [9]:
titles_path = '/kaggle/input/datasets/netflix-inc/netflix-prize-data/movie_titles.csv'

with open(titles_path, 'r', encoding='latin-1') as f:
    for i in range(8):
        print(f.readline().strip())

1,2003,Dinosaur Planet
2,2004,Isle of Man TT 2004 Review
3,1997,Character
4,1994,Paula Abdul's Get Up & Dance
5,2004,The Rise and Fall of ECW
6,1997,Sick
7,1992,8 Man
8,2004,What the #$*! Do We Know!?


In [10]:
rows = []
with open(titles_path, 'r', encoding='latin-1') as f:
    for line in f:
        movie_id, year, title = line.strip().split(',', 2)
        rows.append((int(movie_id), year, title))

titles = pd.DataFrame(rows, columns=['movie_id', 'year', 'title'])
print(titles.shape)

movie_avg_named = movie_avg.join(titles.set_index('movie_id'))
print(movie_avg_named)

(17770, 3)
          count      mean  year  \
movie_id                          
1           547  3.749543  2003   
2           145  3.558621  2004   
3          2012  3.641153  1997   
4           142  2.739437  1994   
5          1140  3.919298  2004   
6          1019  3.084396  1997   
7            93  2.129032  1992   
8         14910  3.189805  2004   
9            95  2.621053  1991   
10          249  3.180723  2001   
11          198  3.030303  1999   
12          546  3.417582  1947   
13          125  4.552000  2003   
14          118  3.025424  1982   
15          290  3.286207  1988   
16         2699  3.098555  1996   
17         7108  2.903208  2005   
18        10722  3.784369  1994   
19          539  3.324675  2000   
20          116  3.146552  1972   
21          218  3.463303  2002   
22          203  2.246305  2000   
23          615  3.556098  2001   
24         1333  2.993998  1981   
25         1207  3.970174  1997   
26         5861  2.793721  2004   
27       